# Benchmark OCR Vintern-1B tren Kaggle T4

Do thu 200 anh keyframe that, thu batch size 1/4/8/16, ghi lai giay/anh + VRAM dinh.
Notebook nay la ban nhap dung MOT LAN de chot thong so cho pha chay full (khong phai notebook
production).

**Thu tu bat buoc:** chay -> LUU json -> KIEM (assert). Kernel Kaggle ERROR se xoa sach
`/kaggle/working`, nen cell luu PHAI dung truoc cell assert.

In [ ]:
# Cell 1 - Cai dat thu vien, ghim version transformers trong chinh lenh cai.
# Thu moc "<4.50" truoc (doc noi bo ve Vintern/InternVL), hong thi doi sang "<5".
import subprocess
import sys

TRANSFORMERS_SPEC_PRIMARY = "transformers>=4.37,<4.50"
TRANSFORMERS_SPEC_FALLBACK = "transformers>=4.51,<5"

def pip_install(spec):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", spec],
        capture_output=True, text=True,
    )

transformers_spec_used = None
result = pip_install(TRANSFORMERS_SPEC_PRIMARY)
if result.returncode == 0:
    transformers_spec_used = TRANSFORMERS_SPEC_PRIMARY
else:
    print(f"[CAI DAT] Moc {TRANSFORMERS_SPEC_PRIMARY} that bai, thu {TRANSFORMERS_SPEC_FALLBACK}")
    print(result.stderr[-2000:])
    result2 = pip_install(TRANSFORMERS_SPEC_FALLBACK)
    if result2.returncode == 0:
        transformers_spec_used = TRANSFORMERS_SPEC_FALLBACK
    else:
        print(result2.stderr[-2000:])
        raise RuntimeError("Ca hai moc transformers deu cai that bai - dung, khong doan mo model khac")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "timm", "sentencepiece"], check=True)

import transformers
print(f"[CAI DAT] transformers.__version__ = {transformers.__version__}")
print(f"[CAI DAT] Moc spec da dung: {transformers_spec_used}")

In [ ]:
# Cell 2 - R5: CUDA_VISIBLE_DEVICES phai dat TRUOC import torch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch  # import SAU khi da set bien moi truong o tren

assert torch.cuda.is_available(), "Khong thay GPU - kiem tra Kaggle Accelerator da bat T4/P100 chua"
gpu_name = torch.cuda.get_device_name(0)
cap_major, cap_minor = torch.cuda.get_device_capability(0)
print(f"[GPU] Ten: {gpu_name}")
print(f"[GPU] Compute capability: {cap_major}.{cap_minor}")
print(f"[GPU] So GPU nhin thay (phai la 1 vi da gioi han CUDA_VISIBLE_DEVICES=0): {torch.cuda.device_count()}")

In [ ]:
# Cell 3 - Nap Vintern-1B fp16 (T4 KHONG co bf16 phan cung). Thu attn_implementation="sdpa",
# hong thi nap lai khong co tham so do (model card khong xac nhan sdpa duoc ho tro).
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "5CD-AI/Vintern-1B-v3_5"
sdpa_supported = None

try:
    model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="sdpa",
    ).eval().cuda()
    sdpa_supported = True
    print("[MODEL] Nap thanh cong VOI attn_implementation='sdpa'")
except TypeError as e:
    print(f"[MODEL] sdpa khong ho tro (TypeError: {e}) - nap lai khong co tham so do")
    model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).eval().cuda()
    sdpa_supported = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
print(f"[MODEL] sdpa_supported = {sdpa_supported}")
print(f"[MODEL] dtype thuc te cua model: {next(model.parameters()).dtype}")

In [ ]:
# Cell 4 - Cac ham tien xu ly anh, copy nguyen tu system1/research/ocr_asr/ocr/extract_ocr_vintern.py
# (notebook Kaggle phai tu chua, khong import duoc file repo).
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(list(target_ratios), key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % target_aspect_ratio[0]) * image_size,
            (i // target_aspect_ratio[0]) * image_size,
            ((i % target_aspect_ratio[0]) + 1) * image_size,
            ((i // target_aspect_ratio[0]) + 1) * image_size
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) > 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=6):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(img) for img in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

# max_num=6 giu nguyen theo R4 - anh keyframe 576x324 cho dung 3 tile
MAX_NUM = 6
print(f"[TIEN XU LY] MAX_NUM = {MAX_NUM} (giu nguyen theo yeu cau R4)")

In [ ]:
# Cell 5 - HF_TOKEN tu Kaggle Secrets, KHONG hardcode.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("[SECRETS] Da nhan HF_TOKEN tu Kaggle Secrets")
except Exception as e:
    print(f"[SECRETS] [CANH BAO] Khong lay duoc HF_TOKEN: {e} - co the bi loi khi tai model gated")

In [ ]:
# Cell 6 - Do duong dan dataset trong /kaggle/input bang os.walk (slug chua chot - pha 02 dang xac dinh).
# Chon 200 keyframe TRAI DEU: ORDER BY keyframe_id roi lay buoc nhay len//200.
# KHONG lay 200 anh lien nhau - anh cung video giong nhau se ra so do gia.
import glob
import sqlite3

sqlite_path = None
data_root = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".sqlite"):
            sqlite_path = os.path.join(root, f)
            break
    if sqlite_path:
        break

if not sqlite_path:
    raise FileNotFoundError(
        "Khong tim thay file .sqlite nao trong /kaggle/input. "
        "Hay Add Dataset chua metadata/keyframes truoc khi chay."
    )

# data_root la thu muc chua sqlite, hoac thu muc cha neu keyframes/ nam ngang hang
conn = sqlite3.connect(sqlite_path)
conn.row_factory = sqlite3.Row
rows = conn.execute(
    "SELECT keyframe_id, video_id, image_relpath FROM keyframes ORDER BY keyframe_id"
).fetchall()
conn.close()

rows_mau_relpath = rows[0]["image_relpath"]

# Anh co the nam duoi <root>/keyframes/L21/... hoac <root>/L21/... tuy cach dong goi
# dataset, nen thu ca hai roi kiem bang mot file that truoc khi chot.
def _tim_goc_anh(relpath_mau):
    ung_vien = []
    goc_sqlite = os.path.dirname(sqlite_path)
    ung_vien.append((goc_sqlite, relpath_mau))
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "keyframes" in dirs:
            ung_vien.append((root, relpath_mau))
        # dataset chi co L21..L30 o goc: bo tien to "keyframes/" khoi relpath
        if any(d.startswith("L") and d[1:].isdigit() for d in dirs):
            ung_vien.append((root, relpath_mau.split("/", 1)[1] if "/" in relpath_mau else relpath_mau))
    for goc, rel in ung_vien:
        if os.path.isfile(os.path.join(goc, rel)):
            return goc, rel != relpath_mau
    return goc_sqlite, False

data_root, _bo_tien_to = _tim_goc_anh(rows_mau_relpath)
if _bo_tien_to:
    print("[DATASET] Anh nam truc tiep duoi goc (khong co thu muc 'keyframes/') -> bo tien to")

print(f"[DATASET] sqlite: {sqlite_path}")
print(f"[DATASET] data_root (chua thu muc keyframes/): {data_root}")


total = len(rows)
N_SAMPLE = 200
step = max(1, total // N_SAMPLE)
sampled = rows[::step][:N_SAMPLE]

print(f"[DATASET] Tong so keyframe: {total}")
print(f"[DATASET] Buoc nhay lay mau: {step}")
print(f"[DATASET] So anh lay duoc: {len(sampled)}")

# Kiem tra file anh ton tai, loai bo anh thieu
sample_images = []
missing = 0
for row in sampled:
    img_path = os.path.join(data_root, row["image_relpath"])
    if os.path.exists(img_path):
        sample_images.append({
            "keyframe_id": row["keyframe_id"],
            "video_id": row["video_id"],
            "path": img_path,
        })
    else:
        missing += 1

print(f"[DATASET] Anh ton tai: {len(sample_images)} / thieu: {missing}")
assert len(sample_images) >= 100, f"Qua it anh hop le ({len(sample_images)}) de do co y nghia"

sampled_keyframe_ids = [x["keyframe_id"] for x in sample_images]
print(f"[DATASET] 5 keyframe_id dau: {sampled_keyframe_ids[:5]}")
print(f"[DATASET] 5 keyframe_id cuoi: {sampled_keyframe_ids[-5:]}")

In [ ]:
# Cell 7 - Vong lap do batch size 1/4/8/16, tu nho den lon (R day.OOM cua lo nho co truoc).
# Bat OutOfMemoryError TUNG LO, ghi "OOM", empty_cache(), di tiep - KHONG dung ca cell.
import time

QUESTION = ("<image>\nHay trich xuat toan bo van ban xuat hien trong hinh anh nay. "
            "Chi tra ve van ban tho nhan dang duoc, khong them bot giai thich gi khac.")
GEN_CONFIG = dict(max_new_tokens=512, do_sample=False, num_beams=1)

def run_batch(image_batch):
    """Nap + tien xu ly 1 lo anh, goi batch_chat, tra ve list text (co the rong khi loi)."""
    pixel_values_list = []
    num_patches_list = []
    for img_info in image_batch:
        pv = load_image(img_info["path"], max_num=MAX_NUM).to(torch.float16).cuda()
        pixel_values_list.append(pv)
        num_patches_list.append(pv.size(0))
    pixel_values = torch.cat(pixel_values_list, dim=0)
    questions = [QUESTION] * len(num_patches_list)
    with torch.no_grad():
        responses = model.batch_chat(
            tokenizer, pixel_values,
            num_patches_list=num_patches_list,
            questions=questions,
            generation_config=GEN_CONFIG,
        )
    return responses

BATCH_SIZES = [1, 4, 8, 16]
bench_results = {}
all_texts_by_bs = {}

for bs in BATCH_SIZES:
    print(f"\n[DO] === batch_size={bs} ===")
    torch.cuda.reset_peak_memory_stats()
    n_images = min(len(sample_images), 200)
    batches = [sample_images[i:i + bs] for i in range(0, n_images, bs)]

    texts = []
    n_errors = 0
    n_oom = 0
    t0 = time.time()

    for batch in batches:
        try:
            responses = run_batch(batch)
            texts.extend(responses)
        except torch.cuda.OutOfMemoryError:
            print(f"  [OOM] Lo {len(batch)} anh bi OOM tai batch_size={bs} - bo qua, di tiep")
            n_oom += 1
            n_errors += len(batch)
            torch.cuda.empty_cache()
            continue
        except Exception as e:
            print(f"  [LOI] Lo {len(batch)} anh loi khac: {e}")
            n_errors += len(batch)
            continue

    elapsed = time.time() - t0
    n_processed = len(texts)
    sec_per_image = elapsed / n_processed if n_processed > 0 else None
    peak_mem_bytes = torch.cuda.max_memory_allocated()
    peak_mem_gb = peak_mem_bytes / (1024 ** 3)
    n_empty = sum(1 for t in texts if not t or not t.strip())
    empty_ratio = n_empty / n_processed if n_processed > 0 else None

    bench_results[str(bs)] = {
        "batch_size": bs,
        "n_images_attempted": n_images,
        "n_processed": n_processed,
        "n_errors": n_errors,
        "n_oom_batches": n_oom,
        "elapsed_sec": elapsed,
        "sec_per_image": sec_per_image,
        "peak_memory_gb": peak_mem_gb,
        "n_empty_text": n_empty,
        "empty_ratio": empty_ratio,
    }
    all_texts_by_bs[str(bs)] = texts

    print(f"  processed={n_processed}/{n_images} errors={n_errors} oom_batches={n_oom}")
    print(f"  sec/anh={sec_per_image} peak_mem_gb={peak_mem_gb:.2f} empty_ratio={empty_ratio}")

    if empty_ratio is not None and empty_ratio > 0.5:
        print(f"  [CANH BAO] >50% chuoi rong ({empty_ratio:.0%}) - nghi ngo fp16 tran so (NaN)")

In [ ]:
# Cell 8 - LUU (R1: phai dung TRUOC cell KIEM). Ghi bench_ocr_t4.json ra /kaggle/working/
# va in toan bo ra stdout vi khong doc duoc log Kaggle khi dang chay (R8).
import json

# Trich 20 mau chu doc duoc de nguoi soi dau tieng Viet
sample_texts = []
for bs_key, texts in all_texts_by_bs.items():
    for t in texts:
        if t and t.strip():
            sample_texts.append({"batch_size": bs_key, "text": t.strip()})
        if len(sample_texts) >= 20:
            break
    if len(sample_texts) >= 20:
        break

output = {
    "gpu_name": gpu_name,
    "transformers_version": transformers.__version__,
    "transformers_spec_used": transformers_spec_used,
    "sdpa_supported": sdpa_supported,
    "max_num": MAX_NUM,
    "n_sample_images": len(sample_images),
    "sample_step": step,
    "results_by_batch_size": bench_results,
    "sample_texts_20": sample_texts,
}

OUT_PATH = "/kaggle/working/bench_ocr_t4.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"[LUU] Da ghi {OUT_PATH}")
print("[LUU] Noi dung day du:")
print(json.dumps(output, ensure_ascii=False, indent=2))

# Ghi rieng 20 mau chu ra file text de de soi dau tieng Viet
SAMPLES_PATH = "/kaggle/working/sample_texts_20.txt"
with open(SAMPLES_PATH, "w", encoding="utf-8") as f:
    for i, item in enumerate(sample_texts):
        f.write(f"--- mau {i+1} (batch_size={item['batch_size']}) ---\n")
        f.write(item["text"] + "\n\n")
print(f"[LUU] Da ghi {SAMPLES_PATH} ({len(sample_texts)} mau)")

In [ ]:
# Cell 9 - KIEM (R1: dung SAU cell LUU). Assert it nhat 1 lo chay duoc, in bang tong ket.
n_batches_ok = sum(1 for r in bench_results.values() if r["n_processed"] > 0)
assert n_batches_ok >= 1, "Khong co batch_size nao chay duoc - xem log loi o Cell 7"

print("[KIEM] Bang tong ket:")
print(f"{'batch_size':>10} | {'processed':>9} | {'errors':>6} | {'sec/anh':>10} | {'peak_gb':>8} | {'empty_ratio':>11}")
for bs_key, r in bench_results.items():
    spi = f"{r['sec_per_image']:.3f}" if r["sec_per_image"] is not None else "N/A"
    er = f"{r['empty_ratio']:.1%}" if r["empty_ratio"] is not None else "N/A"
    print(f"{r['batch_size']:>10} | {r['n_processed']:>9} | {r['n_errors']:>6} | {spi:>10} | {r['peak_memory_gb']:>8.2f} | {er:>11}")

print(f"\n[KIEM] transformers dung duoc: {transformers.__version__} (spec: {transformers_spec_used})")
print(f"[KIEM] sdpa_supported: {sdpa_supported}")
print(f"[KIEM] So batch_size chay duoc: {n_batches_ok}/{len(bench_results)}")
print("[KIEM] PASS - xem /kaggle/working/bench_ocr_t4.json de lay so lieu day du")